In [1]:
# Соединение с Metastore и S3
import os
from pyspark.sql import SparkSession, functions as F

# Имя каталога
catalog = "lk"

# Доступ к minio
access_key = os.getenv("MINIO_ROOT_USER", "minioadmin")
secret_key = os.getenv("MINIO_ROOT_PASSWORD", "minioadmin")
warehouse = os.getenv("LAKEKEEPER_WAREHOUSE", "mydatalab")

#Настройка каталога в Spark
spark = (
    SparkSession.builder.appName("lakekeeper-iceberg-demo")
    .config("spark.sql.extensions", "org.apache.iceberg.spark.extensions.IcebergSparkSessionExtensions")
    .config(f"spark.sql.catalog.{catalog}", "org.apache.iceberg.spark.SparkCatalog")
    .config(f"spark.sql.catalog.{catalog}.type", "rest")
    .config(f"spark.sql.catalog.{catalog}.uri", "http://127.0.0.1:8181/catalog")
    .config(f"spark.sql.catalog.{catalog}.warehouse", warehouse)
    .config(f"spark.sql.catalog.{catalog}.io-impl", "org.apache.iceberg.aws.s3.S3FileIO")
    .config(f"spark.sql.catalog.{catalog}.s3.endpoint", "http://127.0.0.1:9000")
    .config(f"spark.sql.catalog.{catalog}.s3.path-style-access", "true")
    .config(f"spark.sql.catalog.{catalog}.s3.access-key-id", access_key)
    .config(f"spark.sql.catalog.{catalog}.s3.secret-access-key", secret_key)
    .config("spark.sql.defaultCatalog", catalog)
    .getOrCreate()
)

#Уровень логирования
spark.sparkContext.setLogLevel("WARN")

In [2]:
# Создание Namespace
spark.sql("CREATE NAMESPACE IF NOT EXISTS lk.experiment")

DataFrame[]

In [13]:
spark.sql("""CREATE TABLE IF NOT EXISTS lk.experiment.first_table(
  message STRING)
  USING ICEBERG""")

DataFrame[]

In [14]:
#Добавление строки Dataframe API
from pyspark.sql.types import StructType, StructField, StringType
from pyspark.sql import Row

## Читаем таблицу
df = spark.table("lk.experiment.first_table")
## Создаем строку по схеме 
data=[Row(message="Data Frame API")]
new_row = spark.createDataFrame(data, schema=df.schema)
## Сохраняем фрейм данных в таблицу
new_row.writeTo("lk.experiment.first_table").append()

In [15]:
#Добавление строки SparkQL
spark.sql("""INSERT INTO lk.experiment.first_table(message) VALUES ('sparkQL')""")

DataFrame[]

In [16]:
spark.table("lk.experiment.first_table").orderBy("message").show()

+--------------+
|       message|
+--------------+
|Data Frame API|
|       sparkQL|
+--------------+



In [12]:
spark.sql("DROP TABLE lk.experiment.first_table")

DataFrame[]